In [0]:

bronze = "abfss://bronze@databricktraveljournal.dfs.core.windows.net"
table = "post_bookmarks"
bronze_table_parquet_path = f"{bronze}/{table}"

post_bookmarks_df = spark.read.format("parquet")\
    .load(f"{bronze_table_parquet_path}")

display(post_bookmarks_df)

created_at,id,post_id,user_id,date_type,year,month,day,_rescued_data
null,434,899,209,2024-08-06,2024,8,6,"{""created_at"":""2024-08-06T13:00:38.000Z"",""_file_path"":""abfss://travelsource@databricktraveljournal.dfs.core.windows.net/post_bookmarks/year=2024/month=8/day=6/part-00002-tid-2740839233495099087-3c1cea0d-dda3-408a-ae29-c0870b87780e-1045-2.c000.snappy.parquet""}"
null,435,899,264,2024-08-06,2024,8,6,"{""created_at"":""2024-08-06T14:50:50.000Z"",""_file_path"":""abfss://travelsource@databricktraveljournal.dfs.core.windows.net/post_bookmarks/year=2024/month=8/day=6/part-00002-tid-2740839233495099087-3c1cea0d-dda3-408a-ae29-c0870b87780e-1045-2.c000.snappy.parquet""}"
null,437,899,75,2024-08-06,2024,8,6,"{""created_at"":""2024-08-06T07:51:38.000Z"",""_file_path"":""abfss://travelsource@databricktraveljournal.dfs.core.windows.net/post_bookmarks/year=2024/month=8/day=6/part-00002-tid-2740839233495099087-3c1cea0d-dda3-408a-ae29-c0870b87780e-1045-2.c000.snappy.parquet""}"
null,438,899,263,2024-08-06,2024,8,6,"{""created_at"":""2024-08-06T13:05:07.000Z"",""_file_path"":""abfss://travelsource@databricktraveljournal.dfs.core.windows.net/post_bookmarks/year=2024/month=8/day=6/part-00002-tid-2740839233495099087-3c1cea0d-dda3-408a-ae29-c0870b87780e-1045-2.c000.snappy.parquet""}"
null,441,900,115,2024-08-06,2024,8,6,"{""created_at"":""2024-08-06T08:03:03.000Z"",""_file_path"":""abfss://travelsource@databricktraveljournal.dfs.core.windows.net/post_bookmarks/year=2024/month=8/day=6/part-00002-tid-2740839233495099087-3c1cea0d-dda3-408a-ae29-c0870b87780e-1045-2.c000.snappy.parquet""}"
null,442,900,249,2024-08-06,2024,8,6,"{""created_at"":""2024-08-06T07:06:16.000Z"",""_file_path"":""abfss://travelsource@databricktraveljournal.dfs.core.windows.net/post_bookmarks/year=2024/month=8/day=6/part-00002-tid-2740839233495099087-3c1cea0d-dda3-408a-ae29-c0870b87780e-1045-2.c000.snappy.parquet""}"
null,446,900,267,2024-08-06,2024,8,6,"{""created_at"":""2024-08-06T03:51:35.000Z"",""_file_path"":""abfss://travelsource@databricktraveljournal.dfs.core.windows.net/post_bookmarks/year=2024/month=8/day=6/part-00002-tid-2740839233495099087-3c1cea0d-dda3-408a-ae29-c0870b87780e-1045-2.c000.snappy.parquet""}"
null,447,900,213,2024-08-06,2024,8,6,"{""created_at"":""2024-08-06T05:53:34.000Z"",""_file_path"":""abfss://travelsource@databricktraveljournal.dfs.core.windows.net/post_bookmarks/year=2024/month=8/day=6/part-00002-tid-2740839233495099087-3c1cea0d-dda3-408a-ae29-c0870b87780e-1045-2.c000.snappy.parquet""}"
null,450,900,142,2024-08-06,2024,8,6,"{""created_at"":""2024-08-06T00:07:38.000Z"",""_file_path"":""abfss://travelsource@databricktraveljournal.dfs.core.windows.net/post_bookmarks/year=2024/month=8/day=6/part-00002-tid-2740839233495099087-3c1cea0d-dda3-408a-ae29-c0870b87780e-1045-2.c000.snappy.parquet""}"
null,451,900,263,2024-08-06,2024,8,6,"{""created_at"":""2024-08-06T13:35:22.000Z"",""_file_path"":""abfss://travelsource@databricktraveljournal.dfs.core.windows.net/post_bookmarks/year=2024/month=8/day=6/part-00002-tid-2740839233495099087-3c1cea0d-dda3-408a-ae29-c0870b87780e-1045-2.c000.snappy.parquet""}"


In [0]:
post_bookmarks_df.printSchema()

root
 |-- created_at: string (nullable = true)
 |-- id: long (nullable = true)
 |-- post_id: long (nullable = true)
 |-- user_id: long (nullable = true)
 |-- date_type: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- _rescued_data: string (nullable = true)



## Data Quality

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.functions import col, lit, coalesce
from pyspark.sql.functions import col, coalesce, get_json_object, to_timestamp, lit, when



def transform_to_silver(bronze_df: DataFrame) -> DataFrame:
    df = bronze_df

    # date_type is a real date; created_at is "-" so we skip it
        # date_type is a real date; created_at is "-" so we skip it
    df = df.withColumn("created_at",
                               when(col("created_at").isNull(),
                                    get_json_object(col("_rescued_data"),"$.created_at"))
                                    .otherwise(col("created_at"))
                                    )

    df = (
        df
        # if the year is 2024, replace it with 2026 (keeps month/day/time exactly)
        .withColumn(
            "created_at",
            F.when(
                F.col("created_at").startswith("2024"),
                F.regexp_replace("created_at", r"^2024", "2026")
            ).otherwise(F.col("created_at"))
        )
        .withColumn("created_at", F.to_timestamp("created_at"))
        .withColumn("date_type", F.to_date("created_at"))
    )

    df = (
        df.withColumn("created_at", F.to_timestamp("created_at"))
          .withColumn("date_type", F.to_date("date_type"))
    )

    df = (
        df.withColumn("year",          F.expr("try_cast(year as int)"))
          .withColumn("month",         F.expr("try_cast(month as int)"))
          .withColumn("day",           F.expr("try_cast(day as int)"))
          .withColumn("id",            F.expr("try_cast(id as int)"))
          .withColumn("post_id", F.expr("try_cast(post_id as int)"))
          .withColumn("user_id",  F.expr("try_cast(user_id as int)"))
          .withColumn("year",  F.when(F.col("year") == 2024, F.lit(2026)).otherwise(F.col("year")))

    )
    quality_df = df \
            .withColumn("_is_valid", coalesce(
                (col("id").isNotNull()) &
                (col("post_id").isNotNull()) &
                (col("created_at").isNotNull()) &
                (col("user_id").isNotNull()) &

                lit(True)
            ))
        

        # Separate valid and invalid records
    valid_df = quality_df.filter(col("_is_valid"))
    invalid_df = quality_df.filter(~col("_is_valid"))

        # Log invalid records for investigation
    if invalid_df.count() > 0:

            print(f"Quarantined {invalid_df.count()} invalid records")
    return valid_df.drop("_rescued_data","_is_valid")


df = transform_to_silver(post_bookmarks_df)


In [0]:
df.display()

created_at,id,post_id,user_id,date_type,year,month,day
2026-08-06T13:00:38Z,434,899,209,2026-08-06,2026,8,6
2026-08-06T14:50:50Z,435,899,264,2026-08-06,2026,8,6
2026-08-06T07:51:38Z,437,899,75,2026-08-06,2026,8,6
2026-08-06T13:05:07Z,438,899,263,2026-08-06,2026,8,6
2026-08-06T08:03:03Z,441,900,115,2026-08-06,2026,8,6
2026-08-06T07:06:16Z,442,900,249,2026-08-06,2026,8,6
2026-08-06T03:51:35Z,446,900,267,2026-08-06,2026,8,6
2026-08-06T05:53:34Z,447,900,213,2026-08-06,2026,8,6
2026-08-06T00:07:38Z,450,900,142,2026-08-06,2026,8,6
2026-08-06T13:35:22Z,451,900,263,2026-08-06,2026,8,6


## Deduplication

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def deduplicate_by_key(df, keycolumns, order_column, ascending=False):
    """
    
    Deduplicate a Dataframe by composite key, keeping the row with the highest (or lowest) value in order_column

    Args:
        df: Input DataFrame with duplicates
        key_columns: List of columns forming the composite key
        order_column: Column to break ties (e.g., updated_at)
        ascending: If True, keep the smallest order_column value
    """
    
    order_expr = (
        F.col(order_column).asc() if ascending else F.col(order_column).desc()
    )

    window_spec = Window.partitionBy(*keycolumns).orderBy(order_expr)

    return df.withColumn("rank", F.row_number().over(window_spec)).filter(
        F.col("rank") == 1
    ).drop("rank")

post_bookmarks_df = deduplicate_by_key(df, ["id"], "created_at", ascending=True)



In [0]:
post_bookmarks_df.display()

created_at,id,post_id,user_id,date_type,year,month,day
2026-06-18T21:33:51.287Z,2,13,6,2026-06-18,2026,6,18
2026-06-22T18:11:46.465Z,6,19,11,2026-06-22,2026,6,22
2026-06-23T21:21:24.94Z,9,30,16,2026-06-23,2026,6,23
2026-06-23T21:21:27.982Z,10,28,16,2026-06-23,2026,6,23
2026-06-25T18:37:49.668Z,11,19,17,2026-06-25,2026,6,25
2026-06-23T20:03:19.729Z,257,323,77,2026-06-23,2026,6,23
2026-07-14T18:03:19.846Z,258,325,77,2026-07-14,2026,7,14
2026-07-04T05:03:19.961Z,259,326,77,2026-07-04,2026,7,4
2026-06-22T18:03:20.081Z,260,328,76,2026-06-22,2026,6,22
2026-06-25T07:17:01.797Z,261,329,77,2026-06-25,2026,6,25


## Data Writing

In [0]:
post_bookmarks_df.write.format("delta").mode("overwrite").save("abfss://silver@databricktraveljournal.dfs.core.windows.net/post_bookmarks")

## Delta

In [0]:
%sql

CREATE TABLE IF NOT EXISTS travel_journal_catalog.silver.post_bookmarks 
USING DELTA
LOCATION "abfss://silver@databricktraveljournal.dfs.core.windows.net/post_bookmarks"

In [0]:
%sql
SELECT * FROM travel_journal_catalog.silver.post_bookmarks

created_at,id,post_id,user_id,date_type,year,month,day,_is_valid
2026-06-18T21:33:51.287Z,2,13,6,2026-06-18,2026,6,18,null
2026-06-22T18:11:46.465Z,6,19,11,2026-06-22,2026,6,22,null
2026-06-23T21:21:24.94Z,9,30,16,2026-06-23,2026,6,23,null
2026-06-23T21:21:27.982Z,10,28,16,2026-06-23,2026,6,23,null
2026-06-25T18:37:49.668Z,11,19,17,2026-06-25,2026,6,25,null
2026-06-23T20:03:19.729Z,257,323,77,2026-06-23,2026,6,23,null
2026-07-14T18:03:19.846Z,258,325,77,2026-07-14,2026,7,14,null
2026-07-04T05:03:19.961Z,259,326,77,2026-07-04,2026,7,4,null
2026-06-22T18:03:20.081Z,260,328,76,2026-06-22,2026,6,22,null
2026-06-25T07:17:01.797Z,261,329,77,2026-06-25,2026,6,25,null
